In [0]:
CREATE OR REPLACE VIEW formula1ext.gold.v_driver_standing
AS

-- Base filter
WITH base_results AS (
    SELECT *
    FROM formula1ext.gold.fact_session_results
    WHERE season >= 2010
),

-- Step 1: Driver summary (points + high-level stats)
driver_session_summary AS (
    SELECT
        r.season,
        d.driver_id,
        d.driver_name,
        d.nationality,
        COUNT_IF(r.session_type = 'RACE') AS race_starts,
        SUM(r.points) AS total_points,
        COUNT_IF(r.session_type = 'RACE' AND r.is_win) AS number_of_wins,
        COUNT_IF(r.session_type = 'RACE' AND r.is_podium) AS number_of_podiums
    FROM 
        base_results r
        JOIN formula1ext.gold.dim_drivers d
        ON r.driver_id = d.driver_id
    GROUP BY
        r.season,
        d.driver_id,
        d.driver_name,
        d.nationality
),

-- Step 2: Finish distribution + best finish + timing
driver_finish_stats AS (
    SELECT
        season,
        driver_id,

        -- Best finish (handles >P20 and 0-point drivers)
        MIN(final_position) AS best_finish,

        -- Full distribution (FIA tie-breaking)
        SUM(CASE WHEN final_position = 1 THEN 1 ELSE 0 END) AS p1,
        SUM(CASE WHEN final_position = 2 THEN 1 ELSE 0 END) AS p2,
        SUM(CASE WHEN final_position = 3 THEN 1 ELSE 0 END) AS p3,
        SUM(CASE WHEN final_position = 4 THEN 1 ELSE 0 END) AS p4,
        SUM(CASE WHEN final_position = 5 THEN 1 ELSE 0 END) AS p5,
        SUM(CASE WHEN final_position = 6 THEN 1 ELSE 0 END) AS p6,
        SUM(CASE WHEN final_position = 7 THEN 1 ELSE 0 END) AS p7,
        SUM(CASE WHEN final_position = 8 THEN 1 ELSE 0 END) AS p8,
        SUM(CASE WHEN final_position = 9 THEN 1 ELSE 0 END) AS p9,
        SUM(CASE WHEN final_position = 10 THEN 1 ELSE 0 END) AS p10,
        SUM(CASE WHEN final_position = 11 THEN 1 ELSE 0 END) AS p11,
        SUM(CASE WHEN final_position = 12 THEN 1 ELSE 0 END) AS p12,
        SUM(CASE WHEN final_position = 13 THEN 1 ELSE 0 END) AS p13,
        SUM(CASE WHEN final_position = 14 THEN 1 ELSE 0 END) AS p14,
        SUM(CASE WHEN final_position = 15 THEN 1 ELSE 0 END) AS p15,
        SUM(CASE WHEN final_position = 16 THEN 1 ELSE 0 END) AS p16,
        SUM(CASE WHEN final_position = 17 THEN 1 ELSE 0 END) AS p17,
        SUM(CASE WHEN final_position = 18 THEN 1 ELSE 0 END) AS p18,
        SUM(CASE WHEN final_position = 19 THEN 1 ELSE 0 END) AS p19,
        SUM(CASE WHEN final_position = 20 THEN 1 ELSE 0 END) AS p20,

        -- Timing (only used if distribution is identical)
        MIN(CASE WHEN final_position = 1 THEN round END) AS first_p1_round,
        MIN(CASE WHEN final_position = 2 THEN round END) AS first_p2_round,
        MIN(CASE WHEN final_position = 3 THEN round END) AS first_p3_round

    FROM 
        base_results
    WHERE 
        session_type = 'RACE'
        AND final_position IS NOT NULL
        AND final_position > 0
        AND (status = 'Finished' OR status LIKE '+%')
    GROUP BY
        season, driver_id
)

-- Step 3: Final ranking
SELECT
    dss.season,
    dss.driver_id,
    dss.driver_name,
    dss.nationality,

    RANK() OVER (
        PARTITION BY dss.season
        ORDER BY
            -- 1. Points
            dss.total_points DESC,

            -- 2. Distribution (FIA primary tie-breakers)
            dfs.p1 DESC,
            dfs.p2 DESC,
            dfs.p3 DESC,
            dfs.p4 DESC,
            dfs.p5 DESC,
            dfs.p6 DESC,
            dfs.p7 DESC,
            dfs.p8 DESC,
            dfs.p9 DESC,
            dfs.p10 DESC,
            dfs.p11 DESC,
            dfs.p12 DESC,
            dfs.p13 DESC,
            dfs.p14 DESC,
            dfs.p15 DESC,
            dfs.p16 DESC,
            dfs.p17 DESC,
            dfs.p18 DESC,
            dfs.p19 DESC,
            dfs.p20 DESC,

            -- 3. Best finish (handles >P20 and 0-point ties)
            COALESCE(dfs.best_finish, 999) ASC,

            -- 4. Timing (earliest achievement)
            COALESCE(dfs.first_p1_round, 999) ASC,
            COALESCE(dfs.first_p2_round, 999) ASC,
            COALESCE(dfs.first_p3_round, 999) ASC,

            -- 5. Deterministic fallback
            dss.driver_id ASC
    ) AS standing,

    dss.race_starts,
    dss.total_points,
    dss.number_of_wins,
    dss.number_of_podiums

FROM driver_session_summary dss
LEFT JOIN driver_finish_stats dfs
  ON dss.season = dfs.season
 AND dss.driver_id = dfs.driver_id;